# 📈 Modèle de Hull-White — Implémentation Python Complète

---

> **Hull & White (1990)** — *Pricing Interest Rate Derivative Securities*, Review of Financial Studies

Ce notebook couvre l'implémentation complète du modèle de Hull-White à un facteur :

| # | Module | Contenu |
|---|--------|---------|
| 0 | Installation & imports | Librairies, style graphique |
| 1 | Courbe des taux | Spline cubique, taux forward, visualisation |
| 2 | Classe HullWhiteModel | θ(t), B(t,T), A(t,T), prix ZC analytique |
| 3 | Simulation Monte Carlo | Euler-Maruyama, trajectoires, statistiques |
| 4 | Pricing analytique | Obligations, caplets, caps, swaptions |
| 5 | Calibration | L-BFGS-B sur données de caps de marché |
| 6 | Arbre trinomial | Construction, valorisation, obligations callables |
| 7 | Validation | Reproduction exacte de la courbe, diagnostics |
| 8 | Application pratique | Stress-test, ALM, sensibilités DV01 |
| 9 | Comparaison Vasicek vs HW | Calibration exacte vs approximative |
|10 | Tableau de bord | Récapitulatif complet |

---

**EDS du modèle :**

$$dr(t) = [\theta(t) - a\,r(t)]\,dt + \sigma\,dW(t)$$

avec

$$\theta(t) = \frac{\partial f(0,t)}{\partial t} + a\,f(0,t) + \frac{\sigma^2}{2a}\left(1-e^{-2at}\right)$$

**Propriété fondamentale :** par construction de $\theta(t)$, le modèle reproduit **exactement** la courbe des taux observée sur le marché (condition de no-arbitrage).

## ⚙️ 0. Installation et imports

In [ ]:
# !pip install numpy scipy matplotlib pandas  # décommenter si nécessaire

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize, brentq
from scipy.stats import norm
from scipy.interpolate import CubicSpline
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ── Style graphique unifié ──────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})
NAVY, TEAL, GOLD, RED, GREEN = '#0D1B3E','#1E6F8E','#C9A84C','#C0392B','#1E8449'

print('✅ Librairies chargées avec succès.')

---
## 📊 1. Modélisation de la courbe des taux

La courbe des taux zéro-coupon est l'input fondamental du modèle.
Elle détermine entièrement $\theta(t)$ via le taux forward instantané :

$$P(0,T) = e^{-r(T)\cdot T}, \qquad
f(0,t) = \frac{\partial}{\partial T}\bigl[-\ln P(0,T)\bigr]_{T=t}$$

In [ ]:
# ── Courbe BCE 2024 (exemple réaliste) ─────────────────────────────────────
maturities = np.array([0.25, 0.5, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0, 15.0, 20.0, 30.0])
zero_rates  = np.array([0.0350, 0.0370, 0.0380, 0.0360, 0.0340,
                        0.0320, 0.0310, 0.0300, 0.0300, 0.0290, 0.0280])

# Spline cubique sur r(T)*T = -ln P(0,T)
_spline = CubicSpline(maturities, zero_rates * maturities)

def market_zcb(T):
    """Prix zéro-coupon de marché P(0,T) = exp(-r(T)*T)."""
    T   = np.asarray(T, dtype=float)
    r_T = np.interp(T, maturities, zero_rates)
    return np.exp(-r_T * T)

def forward_rate(t):
    """Taux forward instantané f(0,t) = d/dT[r(T)*T] en T=t."""
    t = np.atleast_1d(np.asarray(t, dtype=float))
    t = np.clip(t, maturities[0], maturities[-1])
    return _spline(t, 1)          # 1ère dérivée de la spline

def d_forward_rate(t, eps=1e-5):
    """df(0,t)/dt par différences finies centrées."""
    return (forward_rate(t + eps) - forward_rate(t - eps)) / (2 * eps)

# ── Visualisation ───────────────────────────────────────────────────────────
T_plot = np.linspace(0.25, 30, 300)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(T_plot, np.interp(T_plot, maturities, zero_rates)*100, color=NAVY, lw=2)
axes[0].scatter(maturities, zero_rates*100, color=GOLD, zorder=5, s=60)
axes[0].set(title='Taux zéro-coupon r(T)', xlabel='Maturité (ans)', ylabel='Taux (%)')

axes[1].plot(T_plot, forward_rate(T_plot)*100, color=TEAL, lw=2)
axes[1].set(title='Taux forward instantané f(0,t)', xlabel='Maturité (ans)', ylabel='Taux (%)')

axes[2].plot(T_plot, market_zcb(T_plot), color=GOLD, lw=2)
axes[2].set(title='Prix zéro-coupon P(0,T)', xlabel='Maturité (ans)', ylabel='Prix')

plt.suptitle('Courbe des taux BCE 2024 — Inputs de Hull-White', fontsize=13,
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'Taux 10 ans     : {np.interp(10, maturities, zero_rates)*100:.2f}%')
print(f'Prix ZC 10 ans  : {market_zcb(10):.5f}')
print(f'Forward f(0,5)  : {float(forward_rate(5))*100:.4f}%')

---
## 🏗️ 2. La classe `HullWhiteModel`

Implémentation objet complète avec toutes les formules analytiques.

$$B(t,T) = \frac{1-e^{-a(T-t)}}{a}$$

$$\ln A(t,T) = \ln\frac{P(0,T)}{P(0,t)} + B(t,T)\,f(0,t)
- \frac{\sigma^2}{4a}\left(1-e^{-2at}\right)B(t,T)^2$$

$$\boxed{P(t,T) = A(t,T)\cdot e^{-B(t,T)\,r(t)}}$$

In [ ]:
class HullWhiteModel:
    """
    Modèle de Hull-White à un facteur.
    EDS : dr(t) = [θ(t) - a·r(t)] dt + σ·dW(t)

    Paramètres
    ----------
    a         : vitesse de mean-reversion (> 0)
    sigma     : volatilité instantanée    (> 0)
    maturities: maturités de la courbe de marché
    zero_rates: taux zéro-coupon correspondants
    """

    def __init__(self, a, sigma, maturities, zero_rates):
        assert a > 0 and sigma > 0, "a et sigma doivent être > 0."
        self.a      = a
        self.sigma  = sigma
        self.mats   = np.asarray(maturities)
        self.rates  = np.asarray(zero_rates)
        self._spl   = CubicSpline(self.mats, self.rates * self.mats)

    # ── Courbe de marché ────────────────────────────────────────────────────
    def market_zcb(self, T):
        T   = np.asarray(T, dtype=float)
        r_T = np.interp(T, self.mats, self.rates)
        return np.exp(-r_T * T)

    def forward_rate(self, t):
        t = np.atleast_1d(np.asarray(t, dtype=float))
        t = np.clip(t, self.mats[0], self.mats[-1])
        return self._spl(t, 1)

    def d_forward_rate(self, t, eps=1e-5):
        return (self.forward_rate(t+eps) - self.forward_rate(t-eps)) / (2*eps)

    # ── Paramètre θ(t) ──────────────────────────────────────────────────────
    def theta(self, t):
        """
        θ(t) = df(0,t)/dt + a·f(0,t) + σ²/(2a)·(1−e^{−2at})

        Garantit la calibration exacte à la courbe de marché observée.
        """
        a, sigma = self.a, self.sigma
        adj = (sigma**2 / (2*a)) * (1 - np.exp(-2*a*t))
        return self.d_forward_rate(t) + a * self.forward_rate(t) + adj

    # ── Composantes analytiques ─────────────────────────────────────────────
    def B(self, t, T):
        """B(t,T) = (1−e^{−a(T−t)})/a  [sensibilité au taux court]"""
        return (1 - np.exp(-self.a*(T-t))) / self.a

    def A(self, t, T):
        """Facteur d'ajustement garantissant la cohérence avec le marché."""
        a, sigma = self.a, self.sigma
        t     = np.asarray(t, dtype=float)
        B_val = self.B(t, T)
        t_safe = np.maximum(t, 1e-9)
        log_A = (np.log(self.market_zcb(T) / self.market_zcb(t_safe))
                 + B_val * np.squeeze(self.forward_rate(t_safe))
                 - (sigma**2/(4*a)) * (1 - np.exp(-2*a*t)) * B_val**2)
        return np.exp(log_A)

    # ── Prix obligations ────────────────────────────────────────────────────
    def zcb_price(self, r_t, t, T):
        """P(t,T) = A(t,T)·exp(−B(t,T)·r(t))"""
        return self.A(t,T) * np.exp(-self.B(t,T) * r_t)

    # ── Propriétés ──────────────────────────────────────────────────────────
    def half_life(self):
        """Demi-vie d'un choc : ln(2)/a (années)."""
        return np.log(2) / self.a

    def asymptotic_std(self):
        """Écart-type asymptotique de r(t) : σ/√(2a)."""
        return self.sigma / np.sqrt(2*self.a)

    def conditional_var(self, t):
        """Var[r(t)|r(0)] = σ²/(2a)·(1−e^{−2at})."""
        return (self.sigma**2/(2*self.a)) * (1 - np.exp(-2*self.a*t))

    def summary(self):
        print('='*50)
        print('    MODÈLE DE HULL-WHITE — PARAMÈTRES')
        print('='*50)
        print(f'  a (mean-reversion)   : {self.a:.6f}')
        print(f'  σ (volatilité)       : {self.sigma:.6f}  ({self.sigma*100:.4f}%)')
        print(f'  Demi-vie du choc     : {self.half_life():.2f} ans')
        print(f'  Écart-type asymp.    : {self.asymptotic_std()*100:.4f}%')
        print('='*50)

# ── Instanciation ───────────────────────────────────────────────────────────
hw = HullWhiteModel(a=0.10, sigma=0.01, maturities=maturities, zero_rates=zero_rates)
hw.summary()

# ── Visualisation de θ(t) ───────────────────────────────────────────────────
T_plt = np.linspace(0.3, 29, 200)
theta_vals = hw.theta(T_plt)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(T_plt, theta_vals*100, color=TEAL, lw=2)
ax.axhline(0, color='gray', lw=0.8, ls='--')
ax.set(title='Fonction θ(t) — Drift de Hull-White calibré à la courbe de marché',
       xlabel='t (ans)', ylabel='θ(t) (%)')
plt.tight_layout()
plt.show()

---
## 🎲 3. Simulation Monte Carlo (Euler-Maruyama)

Schéma de discrétisation :

$$r(t_{i+1}) = r(t_i) + \bigl[\theta(t_i) - a\,r(t_i)\bigr]\Delta t
+ \sigma\sqrt{\Delta t}\,Z_i, \quad Z_i \sim \mathcal{N}(0,1)$$

Distribution exacte conditionnelle :
$r(t)\,|\,r(0) \sim \mathcal{N}\!\left(\mu(t),\; \frac{\sigma^2}{2a}(1-e^{-2at})\right)$

In [ ]:
def simulate_paths(hw, r0, T, n_steps, n_paths, seed=42):
    """
    Simule des trajectoires de taux par schéma d'Euler-Maruyama.
    Retourne : times (n_steps+1,), paths (n_paths, n_steps+1)
    """
    np.random.seed(seed)
    dt      = T / n_steps
    times   = np.linspace(0, T, n_steps + 1)
    paths   = np.zeros((n_paths, n_steps + 1))
    paths[:, 0] = r0
    sqrt_dt = np.sqrt(dt)
    a, sigma = hw.a, hw.sigma

    for i, t in enumerate(times[:-1]):
        th = float(hw.theta(np.array([t])))
        dW = np.random.normal(0, sqrt_dt, n_paths)
        paths[:, i+1] = paths[:, i] + (th - a*paths[:, i])*dt + sigma*dW

    return times, paths

# ── Simulation ───────────────────────────────────────────────────────────────
r0 = 0.03
times, paths = simulate_paths(hw, r0=r0, T=10, n_steps=120, n_paths=1000, seed=42)

mean_p = np.mean(paths, axis=0)
p5, p25, p75, p95 = [np.percentile(paths, q, axis=0) for q in [5,25,75,95]]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Trajectoires
ax = axes[0]
for i in range(100):
    ax.plot(times, paths[i]*100, alpha=0.12, lw=0.7, color=TEAL)
ax.plot(times, mean_p*100, color=NAVY, lw=2.5, label='Moyenne')
ax.fill_between(times, p5*100, p95*100, alpha=0.18, color=RED,  label='IC 90%')
ax.fill_between(times, p25*100,p75*100, alpha=0.28, color=GOLD, label='IC 50%')
ax.axhline(r0*100, color='gray', lw=1, ls='--', label=f'r₀={r0*100:.1f}%')
ax.set(title='Hull-White — 1000 trajectoires Monte Carlo',
       xlabel='Temps (ans)', ylabel='Taux court (%)')
ax.legend(fontsize=9)

# Distribution à T=10
ax = axes[1]
final = paths[:, -1]*100
ax.hist(final, bins=50, color=TEAL, alpha=0.7, density=True, label='Simulation')
std_t = np.sqrt(hw.conditional_var(10))*100
x_g   = np.linspace(final.min(), final.max(), 200)
ax.plot(x_g, norm.pdf(x_g, final.mean(), std_t), color=NAVY, lw=2.5,
        label=f'Théorique N(μ,σ²)')
ax.axvline(final.mean(), color=GOLD, lw=2, ls='--',
           label=f'μ = {final.mean():.2f}%')
ax.set(title='Distribution de r(10) — Simulation vs Théorie',
       xlabel='Taux court (%)', ylabel='Densité')
ax.legend(fontsize=9)

plt.suptitle('Simulation Monte Carlo — Modèle Hull-White',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Moyenne simulée  : {final.mean():.4f}%  |  Écart-type : {final.std():.4f}%')
print(f'Taux négatifs    : {(final < 0).mean()*100:.2f}% des trajectoires')

---
## 💹 4. Pricing analytique — Obligations, Caplets, Caps, Swaptions

**Caplet (formule de type Black) :**
$$\text{Caplet}(T_i, T_{i+1}, K) = P(0,T_{i+1})\bigl[F\,N(d_1) - K\,N(d_2)\bigr]$$
$$\sigma_P = \sigma\,B(T_i,T_{i+1})\sqrt{\frac{1-e^{-2aT_i}}{2a}}$$

In [ ]:
# ── Obligation à taux fixe ───────────────────────────────────────────────────
def price_fixed_bond(hw, r0, coupon_rate, face, maturity, freq=1):
    """Prix d'une obligation à taux fixe = Σ CF_i · P(0, t_i)."""
    t_pay  = np.arange(1/freq, maturity + 1e-9, 1/freq)
    coupon = coupon_rate * face / freq
    return sum(
        (coupon + (face if abs(t - maturity) < 1e-9 else 0.0)) * hw.zcb_price(r0, 0.0, t)
        for t in t_pay
    )

# ── Caplet ──────────────────────────────────────────────────────────────────
def price_caplet(hw, t_start, t_end, K):
    """Prix d'un caplet sur [t_start, t_end], strike K (formule HW analytique)."""
    a, sigma = hw.a, hw.sigma
    delta    = t_end - t_start
    P_s      = hw.market_zcb(t_start)
    P_e      = hw.market_zcb(t_end)
    F        = (P_s/P_e - 1) / delta          # taux forward
    B_val    = hw.B(t_start, t_end)
    sigma_P  = sigma * B_val * np.sqrt((1 - np.exp(-2*a*t_start)) / (2*a))
    if sigma_P < 1e-10:
        return max(F - K, 0.0) * delta * P_e
    d1 = (np.log((1+F*delta)/(1+K*delta)) + 0.5*sigma_P**2) / sigma_P
    d2 = d1 - sigma_P
    return P_e * ((1+F*delta)*norm.cdf(d1) - (1+K*delta)*norm.cdf(d2))

# ── Cap = somme de caplets ───────────────────────────────────────────────────
def price_cap(hw, payment_dates, K):
    """Prix d'un cap = Σ caplets."""
    return sum(
        price_caplet(hw, payment_dates[i], payment_dates[i+1], K)
        for i in range(len(payment_dates)-1)
    )

# ── Swaption européenne (décomposition en options sur ZC) ────────────────────
def price_swaption(hw, r0, T_option, swap_tenor, K, freq=1):
    """Prix d'une swaption payeur-fixe européenne."""
    a, sigma = hw.a, hw.sigma
    pay_times = np.arange(T_option+1/freq, T_option+swap_tenor+1e-9, 1/freq)
    c         = np.full(len(pay_times), K/freq)
    c[-1]    += 1.0                              # remboursement nominal
    B_T       = hw.B(T_option, pay_times)
    V         = (sigma**2/(2*a)) * (1 - np.exp(-2*a*T_option))
    P_T       = hw.zcb_price(r0, 0, T_option)
    P_Ti      = np.array([hw.zcb_price(r0, 0, Ti) for Ti in pay_times])
    A_Ti      = hw.A(T_option, pay_times)

    def bond_val(r_star):
        return np.dot(c, A_Ti * np.exp(-B_T * r_star)) - 1.0

    try:
        r_star = brentq(bond_val, -0.3, 0.8, xtol=1e-10)
    except ValueError:
        return np.nan

    price = 0.0
    for i, Ti in enumerate(pay_times):
        K_i     = A_Ti[i] * np.exp(-B_T[i] * r_star)
        sigma_i = np.sqrt(V) * B_T[i]
        if sigma_i < 1e-10:
            continue
        d1 = (np.log(P_Ti[i]/(K_i*P_T)) + 0.5*sigma_i**2) / sigma_i
        d2 = d1 - sigma_i
        price += c[i] * (P_Ti[i]*norm.cdf(d1) - K_i*P_T*norm.cdf(d2))
    return price

# ── Résultats ────────────────────────────────────────────────────────────────
r0 = 0.03
b1  = price_fixed_bond(hw, r0, coupon_rate=0.05, face=100, maturity=10)
b2  = price_fixed_bond(hw, r0, coupon_rate=0.03, face=100, maturity=5)
c1  = price_cap(hw, np.arange(0, 5.01, 0.25), K=0.04)
s1  = price_swaption(hw, r0, T_option=2.0, swap_tenor=5.0, K=0.035)

print('═'*55)
print('   RÉSULTATS DU PRICING ANALYTIQUE')
print('═'*55)
print(f'  Obligation 5% 10 ans    : {b1:.4f}')
print(f'  Obligation 3%  5 ans    : {b2:.4f}')
print(f'  Cap 5 ans strike 4%     : {c1*100:.4f}% du nominal')
print(f'  Swaption 2Y×5Y K=3.5%  : {s1*100:.4f}% du nominal')
print('═'*55)

# ── Surface des prix de caps ─────────────────────────────────────────────────
strikes  = np.linspace(0.02, 0.06, 12)
mats_cap = np.array([1, 2, 3, 5, 7, 10])
surf     = np.zeros((len(mats_cap), len(strikes)))
for i, m in enumerate(mats_cap):
    for j, K in enumerate(strikes):
        surf[i,j] = price_cap(hw, np.arange(0, m+1e-9, 0.25), K)*100

fig, ax = plt.subplots(figsize=(10,5), subplot_kw={'projection':'3d'})
S, M = np.meshgrid(strikes*100, mats_cap)
ax.plot_surface(M, S, surf, cmap='viridis', alpha=0.85)
ax.set(xlabel='Maturité (ans)', ylabel='Strike (%)', zlabel='Prix cap (%)',
       title='Surface des prix de caps — Hull-White')
plt.tight_layout()
plt.show()

---
## 🔧 5. Calibration du modèle

Détermination de $(a^*, \sigma^*)$ par minimisation des erreurs quadratiques :

$$(a^*,\sigma^*) = \arg\min_{a,\sigma}
\sum_i w_i\left[\text{Cap}_{\text{HW}}(a,\sigma,T_i,K_i)
- \text{Cap}_{\text{marché}}(T_i,K_i)\right]^2$$

In [ ]:
# ── Données de marché (caps) ─────────────────────────────────────────────────
cap_data = [
    (1,  0.035, 0.00100),
    (2,  0.035, 0.00250),
    (3,  0.035, 0.00480),
    (5,  0.035, 0.00850),
    (7,  0.035, 0.01100),
    (10, 0.035, 0.01450),
    (2,  0.040, 0.00120),
    (5,  0.040, 0.00620),
    (10, 0.040, 0.01050),
]

def objective(params):
    a, sigma = params
    if a <= 1e-5 or sigma <= 1e-5:
        return 1e10
    try:
        m     = HullWhiteModel(a, sigma, maturities, zero_rates)
        error = 0.0
        for (mat, K, mkt) in cap_data:
            d   = np.arange(0, mat+1e-9, 0.25)
            mdl = price_cap(m, d, K)
            w   = 1.0 / max(mkt, 1e-6)         # pondération inversement proportionnelle
            error += w * (mdl - mkt)**2
        return error
    except Exception:
        return 1e10

# ── Multi-départ pour éviter les minima locaux ───────────────────────────────
print('⏳ Calibration en cours (multi-départ)...\n')
best_result, best_val = None, np.inf
starts = [[0.10, 0.010], [0.05, 0.015], [0.20, 0.008], [0.30, 0.020]]

for x0 in starts:
    res = minimize(objective, x0=x0, method='L-BFGS-B',
                   bounds=[(1e-4, 3.0), (1e-4, 0.5)],
                   options={'ftol':1e-14, 'gtol':1e-10, 'maxiter':2000})
    if res.fun < best_val:
        best_val, best_result = res.fun, res

a_opt, sigma_opt = best_result.x
hw_cal = HullWhiteModel(a_opt, sigma_opt, maturities, zero_rates)
print('✅ Calibration terminée.\n')
hw_cal.summary()

# ── Tableau comparatif ───────────────────────────────────────────────────────
print('\n📋 Marché vs Modèle calibré :')
print(f'  {"Mat":>4} {"K":>7} {"Marché":>10} {"Modèle":>10} {"Erreur (bp)":>12}')
print('  ' + '-'*46)
errors_bp = []
for (mat, K, mkt) in cap_data:
    d    = np.arange(0, mat+1e-9, 0.25)
    mdl  = price_cap(hw_cal, d, K)
    ebp  = abs(mdl-mkt)*10000
    errors_bp.append(ebp)
    print(f'  {mat:>4.0f}Y {K*100:>6.1f}% {mkt*100:>9.4f}% {mdl*100:>9.4f}% {ebp:>10.2f}bp')
print('  ' + '-'*46)
print(f'  Erreur max : {max(errors_bp):.2f}bp  |  Erreur moyenne : {np.mean(errors_bp):.2f}bp')

# ── Courbe de convergence ────────────────────────────────────────────────────
a_range     = np.linspace(0.02, 0.5, 40)
sigma_range = np.linspace(0.005, 0.03, 40)
A_grid, S_grid = np.meshgrid(a_range, sigma_range)
Z_grid = np.vectorize(lambda a,s: np.log10(objective([a,s])+1e-12))(A_grid, S_grid)

fig, ax = plt.subplots(figsize=(9,5))
cp = ax.contourf(A_grid, S_grid*100, Z_grid, levels=25, cmap='RdYlGn_r')
plt.colorbar(cp, ax=ax, label='log₁₀(Objectif)')
ax.scatter([a_opt], [sigma_opt*100], color='red', s=120, zorder=6,
           marker='*', label=f'Optimum (a={a_opt:.4f}, σ={sigma_opt*100:.4f}%)')
ax.set(title='Surface de calibration — Fonction objectif',
       xlabel='Paramètre a', ylabel='Paramètre σ (%)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
## 🌳 6. Arbre Trinomial de Hull-White

Pour les produits avec exercice anticipé (obligations callables, swaptions bermudiennes).

**Pas de l'arbre :** $\Delta r = \sigma\sqrt{3\Delta t}$

**Probabilités de transition :**
$$p_u = \tfrac{1}{6} + \tfrac{V^2+V}{2}, \quad
p_m = \tfrac{2}{3} - V^2, \quad
p_d = \tfrac{1}{6} + \tfrac{V^2-V}{2}, \quad
V = -a\,j\,\Delta t$$

In [ ]:
class TrinomialTree:
    """
    Arbre trinomial de Hull-White (Hull & White 1994).
    Valorise les dérivés de taux avec exercice anticipé.
    """
    def __init__(self, hw, T, n_steps):
        self.hw = hw
        self.T, self.N = T, n_steps
        self.dt = T / n_steps
        self.dr = hw.sigma * np.sqrt(3 * self.dt)
        self.j_max = max(int(np.ceil(0.184 / (hw.a * self.dt))) + 1, 3)
        self.alphas = np.zeros(n_steps + 1)
        self._calibrate_alphas()

    def _probs(self, j):
        V  = -self.hw.a * j * self.dt
        pu = max(1/6 + (V**2 + V)/2, 0)
        pm = max(2/3 - V**2,         0)
        pd = max(1/6 + (V**2 - V)/2, 0)
        s  = pu + pm + pd
        return pu/s, pm/s, pd/s

    def rate(self, i, j):
        """Taux court au nœud (i, j)."""
        idx = j + self.j_max
        if 0 <= idx < 2*self.j_max + 1:
            return j * self.dr + self.alphas[i]
        return np.nan

    def _calibrate_alphas(self):
        """Calibre les alphas pour reproduire les prix ZC de marché."""
        dt, j_max = self.dt, self.j_max
        Q = {0: 1.0}   # Arrow-Debreu : {j: prix}
        for i in range(self.N):
            P_target = self.hw.market_zcb((i+1)*dt)
            # Alpha tel que Σ Q[j]*exp(-(j*dr+alpha)*dt) = P_target
            sum_Q = sum(q * np.exp(-j*self.dr*dt) for j,q in Q.items())
            self.alphas[i] = (-np.log(P_target) - np.log(sum_Q))/dt if sum_Q > 0 else 0.0
            # Propagation Arrow-Debreu
            Q_new = {}
            for j, q in Q.items():
                r_ij = j*self.dr + self.alphas[i]
                disc = np.exp(-r_ij*dt)
                pu, pm, pd = self._probs(j)
                for dj, prob in [(+1,pu),(0,pm),(-1,pd)]:
                    jn = j + dj
                    if abs(jn) <= j_max:
                        Q_new[jn] = Q_new.get(jn, 0.0) + q*disc*prob
            Q = Q_new
        self.alphas[self.N] = self.alphas[self.N-1]  # extension

    def price(self, terminal_payoff_fn, early_exercise_fn=None):
        """
        Backward induction.
        terminal_payoff_fn(j) -> valeur terminale au nœud j
        early_exercise_fn(i, j, cont) -> valeur avec exercice anticipé
        """
        j_max, N, dt = self.j_max, self.N, self.dt
        V = {j: terminal_payoff_fn(j) for j in range(-j_max, j_max+1)}
        for i in range(N-1, -1, -1):
            V_new = {}
            for j in range(-j_max, j_max+1):
                r_ij = self.rate(i, j)
                if np.isnan(r_ij): continue
                pu, pm, pd = self._probs(j)
                get = lambda jj: V.get(np.clip(jj,-j_max,j_max), 0.0)
                cont = np.exp(-r_ij*dt) * (pu*get(j+1) + pm*get(j) + pd*get(j-1))
                V_new[j] = early_exercise_fn(i,j,cont) if early_exercise_fn else cont
            V = V_new
        return V.get(0, 0.0)


# ── Application : obligation classique vs callable ───────────────────────────
print('🌳 Construction de l\'arbre trinomial (T=10 ans, 60 étapes)...')
tree  = TrinomialTree(hw_cal, T=10, n_steps=60)
face, coup = 100.0, 0.05

# Valeur terminale (remboursement + dernier coupon)
terminal  = lambda j: face * (1 + coup)

# Obligation classique
p_vanilla = tree.price(terminal)

# Callable : émetteur peut rembourser à 100 à partir de l'année 3
call_step = int(3 / tree.dt)
def callable_fn(i, j, cont):
    return min(cont, face) if i >= call_step else cont

p_callable   = tree.price(terminal, early_exercise_fn=callable_fn)
option_value = p_vanilla - p_callable

print(f'\n📊 Résultats arbre trinomial :')
print(f'  Prix obligation classique   : {p_vanilla:.4f}')
print(f'  Prix obligation callable    : {p_callable:.4f}')
print(f'  Valeur option de rachat     : {option_value:.4f}')

# ── Grille des taux de l'arbre ───────────────────────────────────────────────
n_show = min(12, tree.N)
fig, ax = plt.subplots(figsize=(12, 5))
for i in range(n_show+1):
    t_i  = i * tree.dt
    t_nx = (i+1)*tree.dt if i < n_show else None
    for j in range(-tree.j_max, tree.j_max+1):
        r_ij = tree.rate(i, j)
        if np.isnan(r_ij): continue
        ax.scatter(t_i, r_ij*100, color=TEAL, s=18, alpha=0.8, zorder=5)
        if t_nx:
            pu, pm, pd = tree._probs(j)
            for dj in [+1, 0, -1]:
                r_nx = tree.rate(i+1, j+dj)
                if not np.isnan(r_nx):
                    ax.plot([t_i, t_nx], [r_ij*100, r_nx*100], 'lightgray', lw=0.4)
ax.set(title='Arbre trinomial — Grille des taux courts r(i,j)',
       xlabel='Temps (ans)', ylabel='Taux court (%)')
plt.tight_layout()
plt.show()

---
## ✅ 7. Validation — Calibration exacte

**Propriété fondamentale :** par construction de $\theta(t)$, le modèle doit reproduire
**exactement** les prix zéro-coupon de marché. Cette propriété est la condition nécessaire
d'absence d'arbitrage de Hull-White.

In [ ]:
def validate(hw, r0, test_mats):
    records = []
    for T in test_mats:
        mkt  = hw.market_zcb(T)
        mdl  = hw.zcb_price(r0, 0.0, T)
        ebp  = abs(mdl-mkt)*10000
        records.append({
            'Maturité': T,
            'P marché': f'{mkt:.6f}',
            'P modèle': f'{mdl:.6f}',
            'Erreur (bp)': f'{ebp:.4f}',
            'r marché (%)': f'{-np.log(mkt)/T*100:.4f}',
            'r modèle (%)': f'{-np.log(mdl)/T*100:.4f}',
        })
    return pd.DataFrame(records)

r0       = 0.03
test_mats = [0.5, 1, 2, 3, 5, 7, 10, 15, 20, 30]
df_val   = validate(hw_cal, r0, test_mats)
print('\n🔍 VALIDATION — Calibration exacte de Hull-White')
print('='*68)
print(df_val.to_string(index=False))

# ── Taux marché vs modèle ───────────────────────────────────────────────────
r_mkt = [-np.log(hw_cal.market_zcb(T))/T*100 for T in test_mats]
r_mdl = [-np.log(hw_cal.zcb_price(r0,0.0,T))/T*100 for T in test_mats]
errs  = [abs(m-d) for m, d in zip(r_mkt, r_mdl)]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
ax = axes[0]
ax.plot(test_mats, r_mkt, 'o-', color=NAVY, lw=2, ms=8, label='Marché')
ax.plot(test_mats, r_mdl, 's--', color=GOLD, lw=2, ms=7, label='Hull-White')
ax.set(title='Taux ZC — Marché vs Hull-White', xlabel='Maturité (ans)', ylabel='Taux (%)')
ax.legend()
ax = axes[1]
ax.bar(test_mats, errs, color=TEAL, alpha=0.8, width=0.7)
ax.axhline(0.001, color=RED, lw=1.5, ls='--', label='Seuil 0.001%')
ax.set(title='Erreur absolue de taux', xlabel='Maturité (ans)', ylabel='|r_mdl - r_mkt| (%)')
ax.legend()
plt.suptitle('Validation — Propriété de calibration exacte', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# ── Validation Monte Carlo ───────────────────────────────────────────────────
print('\n🎲 Validation Monte Carlo (5000 trajectoires, T=10) :')
_, paths_v = simulate_paths(hw_cal, r0=r0, T=10, n_steps=100, n_paths=5000, seed=0)
dt_v  = 10/100
disc  = np.exp(-np.sum(paths_v[:,:-1], axis=1)*dt_v)
p_mc  = np.mean(disc)
p_ana = hw_cal.zcb_price(r0, 0.0, 10.0)
p_mkt = hw_cal.market_zcb(10.0)
print(f'  P(0,10) Monte Carlo  : {p_mc:.6f}')
print(f'  P(0,10) Analytique   : {p_ana:.6f}')
print(f'  P(0,10) Marché       : {p_mkt:.6f}')
print(f'  Écart MC / Analytique : {abs(p_mc-p_ana)*10000:.2f} bp')

---
## 📉 8. Application pratique — Stress-test & ALM

Portefeuille obligataire multi-lignes :
analyse DV01, convexité et simulation ALM sur 5 ans.

In [ ]:
# ── Portefeuille ─────────────────────────────────────────────────────────────
portfolio = [
    {'nom':'OAT 5% 10 ans',  'coupon':0.05, 'maturite':10, 'face':1_000_000},
    {'nom':'BTP 3%  5 ans',  'coupon':0.03, 'maturite': 5, 'face':  500_000},
    {'nom':'Bund 2% 15 ans', 'coupon':0.02, 'maturite':15, 'face':  750_000},
    {'nom':'Gilts 4% 7 ans', 'coupon':0.04, 'maturite': 7, 'face':  250_000},
]
r0 = 0.03

def ptf_value(hw_m, r0_val, ptf, shock_bp=0):
    rs = r0_val + shock_bp/10000
    return sum(price_fixed_bond(hw_m, rs, b['coupon'], b['face'], b['maturite'])
               for b in ptf)

P0    = ptf_value(hw_cal, r0, portfolio)
P_up  = ptf_value(hw_cal, r0, portfolio, +1)
P_dn  = ptf_value(hw_cal, r0, portfolio, -1)
DV01  = (P_dn - P_up) / 2
Cvx   = (P_up + P_dn - 2*P0)

print('═'*55)
print('   ANALYSE DU PORTEFEUILLE OBLIGATAIRE')
print('═'*55)
print(f'  Valeur totale         : {P0:>15,.2f} EUR')
print(f'  DV01 (par 1 bp)       : {DV01:>15,.2f} EUR')
print(f'  Convexité (approx.)   : {Cvx:>15,.2f} EUR')
print('═'*55)

# ── Stress-test ──────────────────────────────────────────────────────────────
shocks = np.arange(-300, 310, 10)
pnl      = [ptf_value(hw_cal, r0, portfolio, s) - P0 for s in shocks]
pnl_lin  = [DV01*(-s) for s in shocks]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
ax = axes[0]
ax.plot(shocks, [p/1e3 for p in pnl],     color=NAVY, lw=2.5, label='P&L Hull-White')
ax.plot(shocks, [p/1e3 for p in pnl_lin], color=RED,  lw=1.5, ls='--', label='P&L linéaire (DV01)')
ax.fill_between(shocks, [p/1e3 for p in pnl], [p/1e3 for p in pnl_lin],
                alpha=0.2, color=GOLD, label='Effet convexité')
ax.axhline(0, color='gray', lw=0.8)
ax.set(title='Stress-test — P&L vs choc parallèle',
       xlabel='Choc (bp)', ylabel='P&L (k€)')
ax.legend(fontsize=9)

ax = axes[1]
vals = {b['nom']: price_fixed_bond(hw_cal, r0, b['coupon'], b['face'], b['maturite'])
        for b in portfolio}
colors_b = [NAVY, TEAL, GOLD, RED]
bars = ax.barh(list(vals.keys()), [v/1e3 for v in vals.values()],
               color=colors_b, alpha=0.85)
ax.set(title='Valeur par ligne de portefeuille', xlabel='Valeur (k€)')
for bar, v in zip(bars, vals.values()):
    ax.text(v/1e3+2, bar.get_y()+bar.get_height()/2,
            f'{v/1e3:,.0f}', va='center', fontsize=9)

plt.suptitle('Gestion des risques — Portefeuille Hull-White', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# ── ALM 5 ans ────────────────────────────────────────────────────────────────
print('\n🏦 Simulation ALM — 1000 scénarios sur 5 ans')
_, paths_alm = simulate_paths(hw_cal, r0=r0, T=5, n_steps=60, n_paths=1000, seed=99)
pnl_alm = [
    sum(price_fixed_bond(hw_cal, paths_alm[k,-1], b['coupon'], b['face'], b['maturite']-5)
        for b in portfolio if b['maturite']>5) - P0
    for k in range(1000)
]
var99 = np.percentile(pnl_alm, 1)
es99  = np.mean([p for p in pnl_alm if p <= var99])

fig, ax = plt.subplots(figsize=(10,4))
ax.hist(pnl_alm, bins=50, color=TEAL, alpha=0.75, density=True)
ax.axvline(var99, color=RED, lw=2, ls='--', label=f'VaR 99% : {var99/1e3:,.0f} k€')
ax.axvline(np.mean(pnl_alm), color=GOLD, lw=2, label=f'Moyenne : {np.mean(pnl_alm)/1e3:,.0f} k€')
ax.set(title='Distribution P&L ALM à 5 ans — 1000 scénarios',
       xlabel='P&L (EUR)', ylabel='Densité')
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

print(f'  VaR 99%  : {var99/1e3:,.0f} k€')
print(f'  ES  99%  : {es99/1e3:,.0f} k€')

---
## 📐 9. Comparaison Vasicek vs Hull-White

Illustration quantitative de la différence fondamentale :
Vasicek a une moyenne $b$ **constante**, Hull-White a $\theta(t)/a$ **dépendante du temps**.

In [ ]:
class VasicekModel:
    """dr = a(b−r)dt + σ dW"""
    def __init__(self, a, b, sigma):
        self.a, self.b, self.sigma = a, b, sigma

    def zcb_price(self, r0, t, T):
        a, b, sigma = self.a, self.b, self.sigma
        tau = T - t
        B   = (1 - np.exp(-a*tau)) / a
        A   = np.exp((B-tau)*(a**2*b - sigma**2/2)/a**2 - sigma**2*B**2/(4*a))
        return A * np.exp(-B*r0)

# Mêmes a et sigma, b = taux 10 ans de marché
b_vas  = float(np.interp(10, maturities, zero_rates))
vasicek = VasicekModel(a=a_opt, b=b_vas, sigma=sigma_opt)
r0 = 0.03

T_vals = np.array([0.5,1,2,3,5,7,10,15,20,30])
p_mkt = [hw_cal.market_zcb(T)       for T in T_vals]
p_hw  = [hw_cal.zcb_price(r0,0,T)   for T in T_vals]
p_vas = [vasicek.zcb_price(r0,0,T)  for T in T_vals]

err_hw  = [abs(p_hw[i]-p_mkt[i])*10000  for i in range(len(T_vals))]
err_vas = [abs(p_vas[i]-p_mkt[i])*10000 for i in range(len(T_vals))]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
ax = axes[0]
ax.plot(T_vals, [m*100 for m in p_mkt], 'o-', color=NAVY, lw=2.5, ms=8, label='Marché')
ax.plot(T_vals, [m*100 for m in p_hw],  's--', color=TEAL, lw=2,  ms=7, label='Hull-White')
ax.plot(T_vals, [m*100 for m in p_vas], '^:',  color=RED,  lw=2,  ms=7, label='Vasicek')
ax.set(title='Prix ZC : Marché vs Hull-White vs Vasicek',
       xlabel='Maturité (ans)', ylabel='Prix (%)')
ax.legend()

ax = axes[1]
w = 0.4
pos = np.arange(len(T_vals))
ax.bar(pos-w/2, err_hw,  width=w, color=TEAL, alpha=0.85, label='Hull-White')
ax.bar(pos+w/2, err_vas, width=w, color=RED,  alpha=0.85, label='Vasicek')
ax.set_xticks(pos)
ax.set_xticklabels([f'{T}a' for T in T_vals])
ax.set(title='Erreur de pricing vs marché (bp)',
       xlabel='Maturité', ylabel='Erreur (bp)')
ax.legend()
plt.suptitle('Hull-White vs Vasicek — Calibration exacte vs approximative',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print(f'\n📌 Conclusion :')
print(f'  Vasicek    — Erreur max : {max(err_vas):.2f} bp')
print(f'  Hull-White — Erreur max : {max(err_hw):.4f} bp  ← calibration EXACTE ✅')

---
## 🗂️ 10. Tableau de bord récapitulatif

In [ ]:
print('\n' + '█'*60)
print('   TABLEAU DE BORD — MODÈLE DE HULL-WHITE')
print('█'*60)

print('\n── 1. PARAMÈTRES CALIBRÉS ─────────────────────────────────')
hw_cal.summary()

print('\n── 2. PRICING ANALYTIQUE ──────────────────────────────────')
r0 = 0.03
print(f'  Obligation 5% 10 ans    : {price_fixed_bond(hw_cal,r0,0.05,100,10):.4f}')
print(f'  Obligation 3%  5 ans    : {price_fixed_bond(hw_cal,r0,0.03,100, 5):.4f}')
print(f'  Cap 5 ans strike 4%     : {price_cap(hw_cal, np.arange(0,5.01,0.25), 0.04)*100:.4f}%')
print(f'  Swaption 2Y×5Y K=3.5%  : {price_swaption(hw_cal,r0,2,5,0.035)*100:.4f}%')

print('\n── 3. ARBRE TRINOMIAL ─────────────────────────────────────')
print(f'  Obligation classique 5% 10 ans  : {p_vanilla:.4f}')
print(f'  Obligation callable             : {p_callable:.4f}')
print(f'  Valeur option de rachat         : {option_value:.4f}')

print('\n── 4. RISQUES DU PORTEFEUILLE ─────────────────────────────')
print(f'  Valeur totale   : {P0:>15,.2f} EUR')
print(f'  DV01 (1 bp)     : {DV01:>15,.2f} EUR')
print(f'  VaR 99% 5 ans   : {var99/1e3:>12,.0f} k€')

print('\n── 5. VALIDATION CALIBRATION ──────────────────────────────')
errs_v = [abs(hw_cal.market_zcb(T)-hw_cal.zcb_price(r0,0,T))*10000 for T in [1,5,10,20,30]]
print(f'  Erreur max sur prix ZC : {max(errs_v):.6f} bp  → Calibration exacte ✅')

print('\n── 6. PROPRIÉTÉS ÉCONOMIQUES ──────────────────────────────')
print(f'  Demi-vie du choc         : {hw_cal.half_life():.2f} ans')
print(f'  Écart-type asymptotique  : {hw_cal.asymptotic_std()*100:.4f}%')
print(f'  θ(1 an)  = {float(hw_cal.theta(np.array([1.0])))*100:.4f}%')
print(f'  θ(5 ans) = {float(hw_cal.theta(np.array([5.0])))*100:.4f}%')
print(f'  θ(10 ans)= {float(hw_cal.theta(np.array([10.0])))*100:.4f}%')
print('\n' + '█'*60)

---
## 📚 Références

| Auteurs | Titre | Revue |
|---------|-------|-------|
| **Hull & White (1990)** | Pricing Interest Rate Derivative Securities | *Review of Financial Studies*, Vol. 3, No. 4 |
| **Hull & White (1994a)** | Numerical Procedures for Implementing Term Structure Models I | *Journal of Derivatives*, Vol. 2, No. 1 |
| **Hull & White (1994b)** | Numerical Procedures for Implementing Term Structure Models II | *Journal of Derivatives*, Vol. 2, No. 2 |
| **Vasicek (1977)** | An Equilibrium Characterization of the Term Structure | *Journal of Financial Economics*, Vol. 5 |
| **Brigo & Mercurio (2006)** | Interest Rate Models — Theory and Practice | Springer |
| **Andersen & Piterbarg (2010)** | Interest Rate Modeling (3 volumes) | Atlantic Financial Press |

---
*Notebook réalisé dans le cadre de la présentation du modèle de Hull-White.
Toutes les formules sont issues des articles originaux de Hull & White (1990, 1994).*